# Notebook 11 — Prédire les 12 prochains mois

**Ce que fait cette piste.** Au lieu de prédire le rendement excédentaire du **mois suivant**,
les étapes 11 à 14 prédisent celui des **12 mois suivants**, composé :

$$\text{cible}_{i,t} = \prod_{k=1}^{12}\left(1 + R_{i,t+k}\right) \;-\; \prod_{k=1}^{12}\left(1 + Rf_{t+k}\right)$$

C'est bien un rendement **excédentaire** : les rendements et le taux sans risque sont
capitalisés **séparément** puis soustraits. C'est le seul choix cohérent avec la cible
d'origine (`excess_return` = RET − Rfree) ; utiliser $\prod(1+R)-1$ aurait changé d'horizon
*et* de définition en même temps, rendant les deux pistes incomparables.

⚠️ **Cette piste s'ajoute au pipeline existant, elle ne le remplace pas.** Les étapes 04 à 07
et les notebooks 04 à 10 ne sont ni modifiés ni relancés : ils continuent de tourner sur
l'horizon 1 mois, et leurs fichiers de sortie ne sont jamais écrasés (ceux de cette piste
portent le suffixe `_h12`).

⚠️ **Ce notebook s'ouvre même si une seule des étapes 11 à 14 a été lancée.** Il détecte les
modèles disponibles et signale ceux qui manquent. Les quatre étapes sont indépendantes.

**Plan :**
1. Ce que coûte l'horizon long : combien d'observations, et pourquoi
2. `R²_oos` par modèle et par fenêtre
3. Portefeuilles — cohortes chevauchantes **et** rebalancement annuel
4. Comparaison avec l'horizon 1 mois

---
## ⚠️ Quatre avertissements à ne pas oublier dans le mémoire

### 1. Les observations se **chevauchent**

Deux dates de prévision consécutives du même titre partagent 11 mois de rendement. C'est la
propriété qui gouverne tout le reste de ce notebook, et elle a quatre conséquences :

- **fuite aux frontières des fenêtres** — traitée par un **embargo** de 12 mois retiré à la
  fin du train et de la validation (`fenetres.appliquer_embargo`). Sans lui, la cible des
  derniers mois du train porterait sur des rendements appartenant à la validation ;
- **t-stats gonflées** — les résidus suivent un MA(11). C'est pourquoi la section
  Fama-MacBeth de l'étape 04 **n'est pas reprise ici** : sa règle de lags Newey-West (≈ 6)
  est calibrée pour des rendements mensuels non chevauchants et donnerait des t-stats
  largement surévaluées. Il faudrait au minimum 11 lags (Hansen-Hodrick) ;
- **portefeuilles** — la construction du notebook 08 ne s'applique pas telle quelle
  (section 3) ;
- **R²_oos incomparable entre horizons** (avertissement 2).

### 2. Le `R²_oos` n'est **pas** comparable entre 1 mois et 12 mois

Son dénominateur est $\sum_i y_i^2$, c'est-à-dire l'échelle des rendements réalisés. La
variance d'un rendement sur 12 mois est bien supérieure à celle d'un rendement mensuel : les
deux séries de R² n'ont pas le même dénominateur et **ne se comparent pas**.

Le R²_oos reste utile pour comparer les **modèles entre eux à horizon donné**. Pour comparer
les deux **horizons**, il faut le **rank-IC** (invariant à l'échelle) et le **Sharpe** du
portefeuille (annualisé de la même façon des deux côtés). C'est ce que fait la section 4.

### 3. Les entreprises radiées sont **conservées**, pas supprimées

Une entreprise qui disparaît (faillite, rachat) voit sa position liquidée au dernier
rendement observé, le solde étant placé au taux sans risque ou 0 jusqu'à t+12 — convention
Shumway (1997) pour taux sans risque, paramètre `config.TRAITEMENT_RADIATION`.

Les écarter aurait produit un **biais de survie sévère** : la disparition d'une entreprise
est massivement corrélée à sa performance, donc on aurait supprimé précisément les
observations où le rendement à 12 mois est le plus négatif, et le modèle n'aurait été
entraîné et évalué que sur des titres dont on sait *rétrospectivement* qu'ils ont survécu.

⚠️ **Limite à déclarer** : l'étape 02 supprime les codes CRSP non numériques de `RET`, et le
*delisting return* (`DLRET`) n'est pas dans la base. Le « dernier rendement observé » est
donc le dernier rendement mensuel régulier, pas le rendement de radiation — souvent très
négatif. La convention **sous-estime** la perte, ce qui reste bien préférable à supprimer
l'entreprise.

### 4. Un mois manquant coûte **douze** observations

Un trou au milieu d'un historique invalide toutes les dates de prévision dont la fenêtre
l'enjambe, soit 12 dates — pas une. Ces lignes sont écartées : un trou au milieu traduit
presque toujours un problème de donnée, événement largement indépendant de la performance
future, contrairement à une radiation.

La censure de fin d'échantillon (les 12 derniers mois du panel) est écartée elle aussi, et
**uniformément pour tous les titres** — y compris les radiés. Ne l'appliquer qu'aux titres
vivants aurait laissé, sur les 12 derniers mois, un échantillon composé uniquement
d'entreprises radiées.

La section 1 chiffre chacun de ces trois cas.

---
## 0. Imports et détection des étapes déjà lancées

In [ ]:
import sys
sys.path.append("..")  # config.py, horizon.py, portefeuilles.py, rapports.py sont a la racine

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config
import fenetres
import portefeuilles
import rapports

config.assurer_dossiers()
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

HORIZON = config.HORIZON_PREDICTION_MOIS
CIBLE_LONGUE = config.nom_cible_horizon()
CIBLE_COURTE = 'excess_return'

# Une entree par etape 11 a 14. Le notebook s'accommode de n'importe quel sous-ensemble
# deja execute : les etapes sont independantes et peuvent etre lancees une par une.
ETAPES = [
    ('Regression lineaire', 'regression_lineaire', '11_horizon_lineaire',
     config.FICHIER_PREDICTIONS_REGRESSION_LINEAIRE),
    ('Elastic Net', 'elastic_net', '12_horizon_elastic_net',
     config.FICHIER_PREDICTIONS_ELASTIC_NET),
    ('LightGBM', 'lightgbm', '13_horizon_lightgbm',
     config.FICHIER_PREDICTIONS_LIGHTGBM),
    ('Random Forest', 'random_forest', '14_horizon_random_forest',
     config.FICHIER_PREDICTIONS_RANDOM_FOREST),
]

disponibles, manquants = [], []
for nom, cle, nom_rapport, chemin_1mois in ETAPES:
    chemins = config.fichiers_horizon(cle)
    if chemins['predictions'].exists():
        disponibles.append({'modele': nom, 'cle': cle, 'rapport': nom_rapport,
                            'chemins': chemins, 'predictions_1mois': chemin_1mois})
    else:
        manquants.append(nom)

print(f"Horizon : {HORIZON} mois   |   cible : {CIBLE_LONGUE!r}")
print(f"\nModeles disponibles ({len(disponibles)}/4) : "
      + (", ".join(d['modele'] for d in disponibles) or "aucun"))
if manquants:
    print(f"Modeles non encore lances : {', '.join(manquants)}")
    print("  -> lance le script correspondant (etape11 a etape14) puis re-execute ce notebook.")
if not disponibles:
    raise SystemExit("Lance au moins une des etapes 11 a 14 avant d'ouvrir ce notebook.")

MODELES = [d['modele'] for d in disponibles]

---
## 1. Ce que coûte l'horizon long

Le diagnostic est produit par l'étape 03 au moment du calcul de la cible. Il chiffre les
trois cas d'horizon incomplet décrits dans l'avertissement 4.

In [ ]:
rap03 = rapports.charger('03_panel')

n_observes = rap03.valeur('horizon_n_titres_mois_observes')
n_valide = rap03.valeur('horizon_n_cible_valide')
n_trou = rap03.valeur('horizon_n_perdues_trou_milieu')
n_censure = rap03.valeur('horizon_n_perdues_censure_fin')

resume = pd.DataFrame([
    {'cas': 'Cible calculable', 'titres_mois': n_valide,
     'traitement': 'conservee'},
    {'cas': 'Trou au milieu de l historique', 'titres_mois': n_trou,
     'traitement': 'ecartee (12 dates par trou)'},
    {'cas': 'Censure de fin d echantillon', 'titres_mois': n_censure,
     'traitement': 'ecartee (tous titres, uniformement)'},
])
resume['part_%'] = (resume['titres_mois'] / n_observes * 100).round(1)

print(f"Titres-mois observes au total : {n_observes}")
display(resume)

print(f"\nEntreprises avec au moins un trou au milieu : {rap03.valeur('horizon_n_permno_avec_trou')}")
print(f"Entreprises radiees : {rap03.valeur('horizon_n_permno_radies')}")
if rap03.valeur('horizon_n_rendements_perte_totale'):
    print(f"⚠️ Rendements <= -100 % ramenes au plancher   : "
          f"{rap03.valeur('horizon_n_rendements_perte_totale')}")

if resume.loc[1, 'part_%'] > 10:
    print("\n⚠️ Plus de 10 % des titres-mois sont perdus par des trous au MILIEU des")
    print("   historiques. A ce niveau, les ecarter selectionne les titres a historique")
    print("   parfait, donc plutot les grandes capitalisations stables. Envisage une")
    print("   imputation par le rendement du marche, et declare-le dans le memoire.")

In [ ]:
# Distribution des deux cibles cote a cote. L'ecart d'echelle est ce qui rend les R2_oos
# des deux horizons incomparables (avertissement 2) : regarde les ecarts-types.
panel = pd.read_parquet(config.FICHIER_PANEL_MODELISATION,
                        columns=['annee_mois', CIBLE_COURTE, CIBLE_LONGUE])


ecart_court = panel[CIBLE_COURTE].std()
ecart_long = panel[CIBLE_LONGUE].std()
print(f"Ecart-type : {ecart_court:.4f} (1 mois) contre {ecart_long:.4f} ({HORIZON} mois), "
      f"soit un facteur {ecart_long / ecart_court:.1f}.")
print("-> le denominateur du R2_oos est ~"
      f"{(ecart_long / ecart_court) ** 2:.0f} fois plus grand a {HORIZON} mois : "
      "les R2 des deux horizons ne se comparent PAS (avertissement 2).")

---
## 2. `R²_oos` par modèle

⚠️ **À lire uniquement en comparant les modèles entre eux**, jamais en le rapprochant des R²
du notebook 09 (horizon 1 mois). La comparaison entre horizons se fait en section 4.

In [ ]:
lignes = []
for d in disponibles:
    r = pd.read_parquet(d['chemins']['resultats'])
    lignes.append(r.iloc[0])
resultats = pd.DataFrame(lignes).reset_index(drop=True)

display(resultats[['modele', 'horizon_mois', 'n_fenetres',
                   'r2_oos_train', 'r2_oos_validation', 'r2_oos_test']].round(5))

print("Ecart train - test (indicateur de sur-apprentissage) :")
for _, l in resultats.iterrows():
    print(f"  {l['modele']:22s} : {l['r2_oos_train'] - l['r2_oos_test']:+.5f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
for d in disponibles:
    par_fenetre = pd.read_parquet(d['chemins']['resultats_fenetre'])
    colonne_annee = 'annee_test' if 'annee_test' in par_fenetre.columns else 'fenetre'
    ax.plot(par_fenetre[colonne_annee].astype(str), par_fenetre['r2_oos_test'],
            marker='o', label=d['modele'], linewidth=1.6)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel("Fenetre de test")
ax.set_ylabel(f"R2_oos test ({HORIZON} mois)")
ax.set_title(f"R2_oos hors echantillon, fenetre par fenetre -- horizon {HORIZON} mois")
ax.legend()
ax.grid(alpha=0.25)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

---
## 3. Portefeuilles

⚠️ **La construction du notebook 08 ne s'applique pas telle quelle ici.** Si on moyenne *la
cible à 12 mois* par décile et par mois, puis qu'on annualise en ×12 et √12, ces « rendements
mensuels » sont en réalité des rendements **annuels chevauchants** : deux mois consécutifs
partagent 11 mois de rendement. L'annualisation serait fausse et la volatilité massivement
sous-estimée.

Trois constructions correctes, calculées côte à côte (`config.MODE_PORTEFEUILLE_HORIZON`) :

**① Rebalancement mensuel — construction PRINCIPALE** (`config.CONSTRUCTION_PRINCIPALE_HORIZON`).
Chaque mois on classe les titres en déciles selon la prédiction à 12 mois, on détient **un
mois**, et le rendement réalisé pris en compte est le rendement excédentaire **mensuel**. La
série est mensuelle et non chevauchante, donc annualisable normalement.
C'est **exactement la mécanique du notebook 08** : seul le *signal de classement* change. La
comparaison entre les deux horizons (section 4) devient donc une comparaison **toutes choses
égales par ailleurs** — et c'est pour cela que c'est la construction de référence ici.
⚠️ Elle ignore en revanche l'horizon de détention implicite de la cible : elle liquide tout
au bout d'un mois alors que le modèle prévoit 12 mois. Son turnover est maximal.

**② Cohortes chevauchantes (Jegadeesh-Titman, 1993).** Chaque mois on forme une cohorte
selon la prédiction à 12 mois, et on la détient 12 mois. Le rendement du portefeuille au
mois *t* est la moyenne des 12 cohortes actives, calculé sur les rendements **mensuels**.
C'est la seule des trois qui **respecte l'horizon de détention** de la cible, et elle
correspond à une stratégie réellement implémentable : on investit 1/12 du capital chaque
mois. Référence de la littérature momentum.

**③ Rebalancement annuel.** Un portefeuille formé une fois par an, détenu 12 mois. Aucun
chevauchement, donc statistiquement irréprochable, mais 12 fois moins d'observations : les
t-stats s'en ressentent. À lire comme un contrôle de robustesse.

Les trois s'annualisent différemment (12 périodes par an pour ① et ②, une seule pour ③) —
`calculer_metriques` en tient compte via `nb_periodes_par_an`.

In [ ]:
# Le rendement MENSUEL realise est indispensable aux constructions 'mensuel' et 'cohortes' :
# il vient du PANEL, pas des fichiers de predictions (qui ne portent que la cible longue).
# On en profite pour recuperer la capitalisation, dont la section 3bis a besoin.
COLONNE_CAPI = config.COLONNE_MVEL1_BRUT

colonnes_panel = ['permno', 'annee_mois', CIBLE_COURTE]
if config.PONDERATION_PAR_CAPITALISATION:
    colonnes_panel.append(COLONNE_CAPI)
panel_mensuel = pd.read_parquet(config.FICHIER_PANEL_MODELISATION, columns=colonnes_panel)
panel_mensuel['annee_mois'] = panel_mensuel['annee_mois'].astype(str)

# Quelles constructions calculer, et laquelle sert de reference partout ailleurs.
CONSTRUCTIONS = config.constructions_portefeuille_horizon()
PRINCIPALE = config.CONSTRUCTION_PRINCIPALE_HORIZON

LIBELLES = {
    'mensuel':  'Rebalancement mensuel',
    'cohortes': 'Cohortes (Jegadeesh-Titman)',
    'annuel':   'Rebalancement annuel',
}
# ⚠️ L'annualisation N'EST PAS la meme pour les trois : 'mensuel' et 'cohortes' produisent
# une serie MENSUELLE (12 periodes par an), 'annuel' une observation tous les HORIZON mois.
PERIODES_PAR_AN = {'mensuel': 12, 'cohortes': 12, 'annuel': 12 / HORIZON}

resultats_p = {construction: {} for construction in CONSTRUCTIONS}
predictions_par_modele = {}

for d in disponibles:
    pred = pd.read_parquet(d['chemins']['predictions'])
    pred['annee_mois'] = pred['annee_mois'].astype(str)
    pred = pred.merge(panel_mensuel, on=['permno', 'annee_mois'], how='left')
    predictions_par_modele[d['modele']] = pred

    if 'mensuel' in CONSTRUCTIONS:
        # Deciles refaits chaque mois sur la prediction A 12 MOIS, detention 1 mois,
        # rendement MENSUEL realise. Meme mecanique que le notebook 08.
        resultats_p['mensuel'][d['modele']] = portefeuilles.portefeuille_mensuel(
            pred, colonne_prediction='prediction',
            colonne_rendement_mensuel=CIBLE_COURTE, nb_deciles=config.NB_DECILES)

    if 'cohortes' in CONSTRUCTIONS:
        resultats_p['cohortes'][d['modele']] = portefeuilles.portefeuille_cohortes(
            pred, colonne_prediction='prediction',
            colonne_rendement_mensuel=CIBLE_COURTE,
            horizon=HORIZON, nb_deciles=config.NB_DECILES)

    if 'annuel' in CONSTRUCTIONS:
        # Seule construction a consommer directement la cible LONGUE : elle peut se le
        # permettre puisqu'elle espace ses observations de HORIZON mois.
        resultats_p['annuel'][d['modele']] = portefeuilles.portefeuille_annuel(
            pred, colonne_prediction='prediction', colonne_cible=CIBLE_LONGUE,
            horizon=HORIZON, nb_deciles=config.NB_DECILES)

print(f"Constructions calculees : {', '.join(LIBELLES[c] for c in CONSTRUCTIONS)}")
print(f"Construction PRINCIPALE : {LIBELLES[PRINCIPALE]}")
print()
for d in disponibles:
    detail = "  |  ".join(
        f"{len(resultats_p[c][d['modele']]['rendement_long_short'].dropna()):4d} obs ({c})"
        for c in CONSTRUCTIONS)
    print(f"{d['modele']:22s} : {detail}")

In [ ]:
lignes = []
for construction in CONSTRUCTIONS:
    for modele in MODELES:
        m = portefeuilles.calculer_metriques(
            resultats_p[construction][modele]['rendement_long_short'],
            nb_periodes_par_an=PERIODES_PAR_AN[construction])
        m.update({'modele': modele, 'construction': LIBELLES[construction]})
        lignes.append(m)

performance = pd.DataFrame(lignes).set_index(['construction', 'modele'])
colonnes = ['rendement_annualise', 'volatilite_annualisee', 'sharpe_ratio',
            'max_drawdown', 't_stat', 'p_value', 'pct_mois_positifs', 'n_mois']
display(performance[colonnes].round(4))

print("Comment lire ce tableau :")
print("  - le rebalancement MENSUEL est la construction principale : c'est la seule dont la")
print("    mecanique soit identique a celle du notebook 08, donc la seule directement")
print("    comparable a l'horizon 1 mois (section 4) ;")
print("  - les COHORTES sont la seule a respecter l'horizon de detention de la cible, et")
print("    ont un turnover bien plus faible que le mensuel -- avantage invisible ici,")
print("    puisque aucun cout de transaction n'est deduit ;")
print("  - le rebalancement ANNUEL repose sur bien moins d'observations : un |t| plus faible")
print("    n'y signifie pas forcement un signal plus faible.")

In [ ]:
fig, axes = plt.subplots(1, len(CONSTRUCTIONS), figsize=(6.5 * len(CONSTRUCTIONS), 4.8),
                         squeeze=False)
axes = axes[0]

for ax, construction in zip(axes, CONSTRUCTIONS):
    largeur = 0.8 / len(MODELES)
    positions = np.arange(config.NB_DECILES)
    for i, modele in enumerate(MODELES):
        moyennes = resultats_p[construction][modele]['rendements_deciles'].mean()
        ax.bar(positions + i * largeur, moyennes.values, width=largeur, label=modele)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(positions + largeur * (len(MODELES) - 1) / 2)
    ax.set_xticklabels(range(1, config.NB_DECILES + 1), fontsize=8)
    ax.set_xlabel("Decile de rendement predit")
    titre = LIBELLES[construction]
    if construction == PRINCIPALE:
        titre += "  [principale]"
    ax.set_title(titre, fontsize=11)
    ax.set_ylabel("Rendement mensuel moyen" if construction != 'annuel'
                  else f"Rendement moyen sur {HORIZON} mois")
    ax.grid(alpha=0.25, axis='y')

axes[0].legend(fontsize=8)
fig.suptitle("Rendement moyen par decile -- les trois constructions", fontsize=12)
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / f"horizon_h{HORIZON}_deciles.png", dpi=150)
plt.show()

In [ ]:
# Richesse cumulee des trois constructions, base 1, sur echelle verticale commune.
TAILLE_FIGURE_RICHESSE = (18, 5)

fig, axes = plt.subplots(1, len(CONSTRUCTIONS), figsize=TAILLE_FIGURE_RICHESSE,
                         squeeze=False, sharey=True)
axes = axes[0]

for ax, construction in zip(axes, CONSTRUCTIONS):
    # ⚠️ 'annuel' n'a qu'une observation tous les HORIZON mois : on marque les points pour
    # que la nature discrete de la serie reste visible et ne se confonde pas avec une
    # courbe mensuelle continue.
    marqueur = 'o' if construction == 'annuel' else None
    for modele in MODELES:
        serie = resultats_p[construction][modele]['rendement_long_short'].dropna()
        dates = pd.to_datetime(serie.index.astype(str), format='%Y%m')
        ax.plot(dates, (1 + serie).cumprod().values, label=modele,
                linewidth=1.6, marker=marqueur, markersize=4)
    ax.axhline(1, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel("Date")
    titre = LIBELLES[construction]
    if construction == PRINCIPALE:
        titre += "  [principale]"
    n_obs = len(resultats_p[construction][MODELES[0]]['rendement_long_short'].dropna())
    titre += f"\n{n_obs} observations"
    ax.set_title(titre, fontsize=11)
    ax.grid(alpha=0.25)
    ax.tick_params(axis='x', rotation=30)

axes[0].set_ylabel("Richesse cumulee (base 1)")
axes[0].legend(fontsize=8)
fig.suptitle(f"Portefeuille long-short -- horizon {HORIZON} mois", fontsize=12)
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / f"horizon_h{HORIZON}_richesse_cumulee.png", dpi=150)
plt.show()

---
## 3bis. Constructions alternatives — de combien le Sharpe bouge-t-il ?

Les portefeuilles ci-dessus sont **équipondérés** et **long-short**, comme partout dans le
projet. Cette section rejoue **exactement les mêmes prédictions** (aucun ré-entraînement)
sous d'autres constructions, pour mesurer ce que ce choix coûte ou rapporte. Deux leviers,
appliqués à la construction principale :

**Pondération par la capitalisation** (`config.PONDERATION_PAR_CAPITALISATION`) — un
portefeuille équipondéré met autant d'argent sur une micro-cap illiquide que sur Apple. Or
c'est précisément chez les petites capitalisations que les modèles d'apprentissage trouvent
l'essentiel de leur signal. Pondérer par la capitalisation (connue en *t*, donc sans
information du futur) reflète où l'argent peut **réellement** être investi. C'est le test de
robustesse de Gu, Kelly & Xiu (2020), et le Sharpe s'effondre généralement au passage. Si
c'est le cas ici, ce n'est pas un échec : c'est un résultat, à rapprocher du notebook 10.

**Long only** (`config.PCT_LONG_ONLY`) — beaucoup d'investisseurs institutionnels ne peuvent
pas vendre à découvert. On achète les *X* % des titres les mieux notés, sans jambe courte.
⚠️ Ce portefeuille porte l'exposition au **marché** (bêta ≈ 1), que le long-short neutralise
en grande partie : son Sharpe se lit contre celui du marché, pas contre celui du long-short.

⚠️ **Aucun coût de transaction n'est déduit**, et les quatre constructions n'ont pas du tout
le même turnover. Le classement ci-dessous est donc **brut** — limite à mentionner
explicitement dans le mémoire.

In [ ]:
# ⚠️ La capitalisation est deja fusionnee dans `predictions_par_modele` (section 3). Ce qui
# manquait ici, c'est le CONTROLE de sa couverture : les portefeuilles ponderes ecartent les
# lignes sans poids, donc ils ne portent pas sur le meme univers que les equiponderes.
# A 12 mois le risque est plus eleve qu'a 1 mois : les mois fantomes ajoutes aux titres
# radies (prolongation Shumway) n'ont PAS de capitalisation.
colonne_poids = COLONNE_CAPI if config.PONDERATION_PAR_CAPITALISATION else None

if colonne_poids is not None:
    pct_manquant = float(
        predictions_par_modele[MODELES[0]][COLONNE_CAPI].isna().mean() * 100)
    print(f"Capitalisation renseignee : {100 - pct_manquant:.1f} % des lignes.")
    if pct_manquant > 5:
        print(f"⚠️ {pct_manquant:.1f} % de capitalisations manquantes. Les portefeuilles")
        print("   ponderes ecartent ces lignes : ils ne portent donc pas exactement sur le")
        print("   meme univers que les equiponderes, et l'ecart de Sharpe melange alors deux")
        print("   effets. A verifier avant de conclure quoi que ce soit.")
else:
    print("config.PONDERATION_PAR_CAPITALISATION = False :")
    print("  seules les variantes equiponderees sont calculees.")

print(f"Long only : les {config.PCT_LONG_ONLY * 100:.0f} % des titres les mieux notes "
      f"(~{config.PCT_LONG_ONLY * config.NB_DECILES:.0f} deciles sur {config.NB_DECILES}).")

In [ ]:
# Les constructions alternatives sont appliquees a la construction PRINCIPALE (rebalancement
# mensuel) : elles consomment donc le rendement MENSUEL realise, comme elle.
colonne_poids = COLONNE_CAPI if config.PONDERATION_PAR_CAPITALISATION else None

comparaisons = {}
for modele in MODELES:
    comparaisons[modele] = portefeuilles.comparer_constructions(
        predictions_par_modele[modele],
        colonne_prediction='prediction',
        colonne_cible=CIBLE_COURTE,
        nb_deciles=config.NB_DECILES,
        colonne_poids=colonne_poids,
        pct_long_only=config.PCT_LONG_ONLY,
        nb_periodes_par_an=12,
    )

performance_constructions = pd.concat(
    {modele: c['performance'] for modele, c in comparaisons.items()},
    names=['modele', 'construction'])

colonnes_alt = ['rendement_annualise', 'volatilite_annualisee', 'sharpe_ratio',
                'max_drawdown','pct_mois_positifs']
display(performance_constructions[colonnes_alt].round(4))

In [ ]:
# Le graphique qui resume la section : un groupe de barres par modele, une barre par
# construction. C'est la figure a mettre dans le memoire pour ce point.
tableau_sharpe = performance_constructions['sharpe_ratio'].unstack('construction')

TAILLE_FIGURE = (11, 5)

# L'etiquette du long only depend de config.PCT_LONG_ONLY : on la reconstruit plutot que
# de la coder en dur, sinon changer PCT_LONG_ONLY casse silencieusement l'ordre ci-dessous.
ETIQUETTE_LONG = f"Long only top {int(round(config.PCT_LONG_ONLY * 100))}%"

# Ordre voulu : les deux long-short d'abord, puis les deux long only.
ORDRE_CONSTRUCTIONS = [
    'Long-short equipondere',
    'Long-short pondere capi',
    f"{ETIQUETTE_LONG} equipondere",
    f"{ETIQUETTE_LONG} pondere capi",
]

# Long-short = bleus (fonce -> clair), long only = rouges (fonce -> clair).
COULEURS_CONSTRUCTIONS = {
    'Long-short equipondere':            '#08519c',
    'Long-short pondere capi':           '#6baed6',
    f"{ETIQUETTE_LONG} equipondere":     '#a50f15',
    f"{ETIQUETTE_LONG} pondere capi":    '#fb6a4a',
}

# Si PONDERATION_PAR_CAPITALISATION = False, seules 2 constructions existent : on ne garde
# que celles reellement presentes, sans planter.
constructions = [c for c in ORDRE_CONSTRUCTIONS if c in tableau_sharpe.columns]

fig, ax = plt.subplots(figsize=TAILLE_FIGURE)
largeur = 0.8 / len(constructions)
positions = np.arange(len(tableau_sharpe))

for i, construction in enumerate(constructions):
    ax.bar(positions + i * largeur, tableau_sharpe[construction].values,
           width=largeur, label=construction,
           color=COULEURS_CONSTRUCTIONS[construction],
           edgecolor='white', linewidth=0.6)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(positions + largeur * (len(constructions) - 1) / 2)
ax.set_xticklabels(tableau_sharpe.index, rotation=15, ha='right')
ax.set_ylabel("Ratio de Sharpe annualise")
ax.set_title(f"Constructions alternatives de portefeuille -- horizon {HORIZON} mois\n"
             "(memes predictions, rebalancement mensuel : seule la construction change)",
             fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.25, axis='y')
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / f"horizon_h{HORIZON}_constructions.png", dpi=150)
plt.show()

REFERENCE = 'Long-short equipondere'
print(f"Ecart moyen de Sharpe par rapport a la construction de reference ({REFERENCE}) :")
for construction in constructions:
    if construction == REFERENCE:
        continue
    ecart = float((tableau_sharpe[construction] - tableau_sharpe[REFERENCE]).mean())
    signe = "au-dessus" if ecart > 0 else "en dessous"
    print(f"  {construction:34s} : {ecart:+.3f}  ({signe} de la reference)")

---
## 4. Comparaison avec l'horizon 1 mois

C'est la section qui répond à la question du mémoire : **prédire à 12 mois vaut-il mieux que
prédire à 1 mois ?**

⚠️ Elle n'utilise **jamais** le `R²_oos` (dénominateurs différents, avertissement 2). Deux
mesures seulement, toutes deux invariantes à l'échelle ou annualisées de façon identique
des deux côtés :

- le **rank-IC**, qui ne dépend que de l'**ordre** des titres ;
- le **Sharpe** du portefeuille long-short, calculé des deux côtés avec la construction à
  **rebalancement mensuel**.

⚠️ Ce second point est le cœur de la comparaison. Les déciles, la fréquence de rebalancement,
la période de détention et l'annualisation sont **strictement identiques** des deux côtés :
la seule chose qui change est le **signal de classement** — prédiction à 1 mois contre
prédiction à 12 mois. Toute différence de Sharpe s'interprète donc directement comme un écart
de qualité du signal, et non comme un artefact de construction. C'est précisément pour rendre
cette comparaison possible que le rebalancement mensuel a été ajouté aux cohortes.

Les modèles à 1 mois ne sont pas relancés : on relit simplement leurs prédictions déjà sur
disque.

In [ ]:
# ⚠️ Les deux horizons passent par le MEME code : rank-IC contre leur cible PROPRE (ce que
# chaque modele pretendait prevoir), portefeuille contre le rendement MENSUEL des deux
# cotes. Seul le signal de classement differe -- c'est tout l'argument de la section.
lignes = []
for d in disponibles:
    sources = [(f'{HORIZON} mois', predictions_par_modele[d['modele']], CIBLE_LONGUE)]
    if d['predictions_1mois'].exists():
        pred_courte = pd.read_parquet(d['predictions_1mois'])
        pred_courte['annee_mois'] = pred_courte['annee_mois'].astype(str)
        sources.append(('1 mois', pred_courte, CIBLE_COURTE))

    for horizon, table, cible_propre in sources:
        ic = portefeuilles.resumer_rank_ic(
            portefeuilles.rank_ic_par_mois(table, 'prediction', cible_propre))
        pf = portefeuilles.evaluer_sous_univers(
            table, 'prediction', CIBLE_COURTE, nb_deciles=config.NB_DECILES)['metriques']
        lignes.append({'modele': d['modele'], 'horizon': horizon,
                       'rank_ic': ic['ic_moyen'],
                       'pct_ic_positif': ic['pct_mois_ic_positif'],
                       'sharpe': pf['sharpe_ratio'],
                       'pct_gagnants': pf['pct_mois_positifs'],
                       't_sharpe': pf['t_stat']})

comparaison = pd.DataFrame(lignes)
ORDRE_H = ['1 mois', f'{HORIZON} mois']
# ⚠️ `pct_ic_positif` = mois ou le CLASSEMENT va dans le bon sens (qualite du signal).
# `pct_gagnants` = mois ou le PORTEFEUILLE gagne de l'argent (regularite economique).
MESURES = {'rank_ic': 'Rank-IC', 'pct_ic_positif': '% mois IC positif',
           'sharpe': 'Sharpe', 'pct_gagnants': '% mois gagnants',
           't_sharpe': 't-stat Sharpe'}

tableau = comparaison.set_index(['modele', 'horizon'])[list(MESURES)].unstack('horizon')
if comparaison['horizon'].nunique() >= 2:
    tableau = tableau.reindex(columns=pd.MultiIndex.from_product([MESURES, ORDRE_H]))
    tableau[('sharpe', 'ecart')] = tableau[('sharpe', ORDRE_H[1])] - tableau[('sharpe', ORDRE_H[0])]
else:
    print("⚠️ Predictions a 1 mois absentes : lance les etapes 04 a 07.")

tableau = tableau.rename(columns=MESURES, level=0)
tableau.columns.names, tableau.index.name = ['', 'horizon'], None
display(tableau.round(3))

In [ ]:
if comparaison['horizon'].nunique() >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for ax, mesure, titre in [
            (axes[0], 'rank_ic', "Rank-IC moyen (invariant a l'echelle)"),
            (axes[1], 'sharpe', "Sharpe du long-short (annualise des 2 cotes)")]:
        # `barres` et non `tableau` : ne pas ecraser le tableau de la cellule precedente.
        barres = comparaison.pivot(index='modele', columns='horizon', values=mesure)
        barres = barres[['1 mois', f'{HORIZON} mois']]
        positions = np.arange(len(barres))
        for i, colonne in enumerate(barres.columns):
            ax.bar(positions + i * 0.35, barres[colonne].values, width=0.35, label=colonne)
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_xticks(positions + 0.175)
        ax.set_xticklabels(barres.index, rotation=20, ha='right')
        ax.set_title(titre, fontsize=11)
        ax.grid(alpha=0.25, axis='y')
    axes[0].legend()
    fig.suptitle("Horizon 1 mois contre horizon long -- mesures comparables uniquement",
                 fontsize=12)
    plt.tight_layout()
    plt.savefig(config.OUTPUTS_DIR / f"horizon_h{HORIZON}_comparaison.png", dpi=150)
    plt.show()
else:
    print("Comparaison graphique indisponible : predictions a 1 mois absentes.")

**Résumé.** Cette piste prédit le rendement excédentaire composé sur 12 mois, en réutilisant
sans les modifier les modèles, hyperparamètres et fenêtres du pipeline principal. Les étapes
04 à 07 et les notebooks 04 à 10 sont restés intacts et continuent de tourner sur l'horizon
1 mois.

**Pour la rédaction :**
- la comparaison entre horizons passe par le **rank-IC** et le **Sharpe**, jamais par le
  `R²_oos` (avertissement 2) ;
- ce Sharpe est calculé des deux côtés avec la construction à **rebalancement mensuel**, la
  seule dont la mécanique soit identique à celle du notebook 08 : seul le signal de
  classement diffère entre les deux horizons ;
- les trois constructions (mensuel, cohortes, annuel) doivent être présentées ensemble : le
  mensuel pour la comparabilité, les cohortes parce qu'elles seules respectent l'horizon de
  détention de la cible, l'annuel comme contrôle sans chevauchement ;
- l'embargo de 12 mois aux frontières des fenêtres doit être mentionné : c'est lui qui
  garantit l'absence de fuite entre train, validation et test (avertissement 1) ;
- la convention de liquidation des titres radiés doit être déclarée
  (`config.TRAITEMENT_RADIATION` : `'taux_sans_risque'`, convention Shumway où la radiation
  est neutre, ou `'zero'`, plus conservatrice où elle coûte le taux sans risque composé),
  ainsi que sa limite (le *delisting return* de CRSP est absent de la base, la perte est
  donc sous-estimée — avertissement 3). Lancer les deux et comparer fait un bon paragraphe
  de robustesse : les deux cohabitent sans se confondre au journal des expériences ;
- le coût en observations de l'horizon long (section 1) doit être chiffré, en distinguant
  trous du milieu et censure de fin d'échantillon (avertissement 4) ;
- l'absence de test Fama-MacBeth pour cet horizon doit être justifiée par le chevauchement
  des observations.

**Limites, et ce que la section 3bis en dit déjà.** Les portefeuilles principaux restent
équipondérés et sans coût de transaction. La section 3bis chiffre le premier point (la
pondération par la capitalisation, et le long only) ; le second reste entier. ⚠️ Le turnover
diffère énormément d'une construction à l'autre — maximal pour le rebalancement mensuel,
douze fois moindre pour l'annuel — et cet écart n'apparaît dans **aucun** des chiffres
ci-dessus. C'est un argument en faveur des cohortes et de l'annuel qu'il faut énoncer
explicitement plutôt que le laisser deviner.

**Pour changer d'horizon** (6 mois, 24 mois) : modifier `config.HORIZON_PREDICTION_MOIS`,
relancer `python scripts/construction_panel.py` puis les étapes 11 à 14 souhaitées.
Les noms de fichiers suivent automatiquement, rien n'est écrasé.